# 1.  Setup


## 1.1 Importing libraries

In [ ]:
# Install any missing libraries
!pip install scikit-learn imbalanced-learn xgboost seaborn matplotlib pandas numpy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import glob
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 1.2 Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.3 Loading the data files

In [ ]:
# Set the folder path
DATA_PATH = '/content/drive/MyDrive/CNS/data/'

csv_files = sorted(glob.glob(DATA_PATH + '*.csv'))
df_list=[]
for file in csv_files:
    temp = pd.read_csv(file, low_memory=False)
    temp.columns = temp.columns.str.strip()
    df_list.append(temp)
    print(f"Loaded {file}")
#turn into dataframes for each one
df_1=df_list[0]
df_2=df_list[1]
df_3=df_list[2]
df_4=df_list[3]
df_5=df_list[4]
df_6=df_list[5]
df_7=df_list[6]
df_8=df_list[7]

Loaded /content/drive/MyDrive/CNS/data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loaded /content/drive/MyDrive/CNS/data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loaded /content/drive/MyDrive/CNS/data/Friday-WorkingHours-Morning.pcap_ISCX.csv
Loaded /content/drive/MyDrive/CNS/data/Monday-WorkingHours.pcap_ISCX.csv
Loaded /content/drive/MyDrive/CNS/data/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loaded /content/drive/MyDrive/CNS/data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loaded /content/drive/MyDrive/CNS/data/Tuesday-WorkingHours.pcap_ISCX.csv
Loaded /content/drive/MyDrive/CNS/data/Wednesday-workingHours.pcap_ISCX.csv


# 2 EDA and Preprocessing

In [ ]:
print(f"Columns of {os.path.basename(csv_files[0])}:")
print(df_list[0].columns)
print("=" * 65)

for i in range(1, len(df_list)): # Start from the second DataFrame
    current_file_name = os.path.basename(csv_files[i])
    print(f"Comparing columns for {current_file_name}:")

    if df_list[0].columns.equals(df_list[i].columns):
        print("   Same columns as the first DataFrame.")
    else:
        print(f"   Columns are different:")
        print(df_list[i].columns)
    print("=" * 65)

Columns of Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv:
Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'P

- Notice that we see two column names that are almost the same (Fwd Header Length and Fwd Header Length.1) further investigation would be conducted on that
- we first check if they have the same values in order to safely drop one of them

In [ ]:
for file in csv_files:
    name = os.path.basename(file)
    temp = pd.read_csv(file, low_memory=False)
    temp.columns = temp.columns.str.strip()

    if 'Fwd Header Length' in temp.columns and 'Fwd Header Length.1' in temp.columns:
        print(f" {name}")

        col1 = temp['Fwd Header Length']
        col2 = temp['Fwd Header Length.1']

        are_equal = col1.equals(col2)
        diff_count = (col1 != col2).sum()
        diff_pct = diff_count / len(temp) * 100

        print(f"   Are values identical     : {are_equal}")
        print(f"   Differing rows           : {diff_count:,}  ({diff_pct:.2f}%)")
        print(f"   Col1 mean                : {col1.mean():.4f}")
        print(f"   Col2 mean                : {col2.mean():.4f}")
        print(f"   Col1 min / max           : {col1.min()} / {col1.max()}")
        print(f"   Col2 min / max           : {col2.min()} / {col2.max()}")

        if diff_count > 0:
            print(f"\n   Sample of differing rows:")
            mask = col1 != col2
            print(temp.loc[mask, ['Fwd Header Length', 'Fwd Header Length.1']].head(10).to_string())
        print()
    else:
        print(f" {name}  →  'Fwd Header Length.1' not found (may already be named differently)")
        print(f"   Columns containing 'Fwd Header' : "
              f"{[c for c in temp.columns if 'Fwd Header' in c]}")
        print()

 Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   Are values identical     : True
   Differing rows           : 0  (0.00%)
   Col1 mean                : 111.5227
   Col2 mean                : 111.5227
   Col1 min / max           : 0 / 39396
   Col2 min / max           : 0 / 39396

 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   Are values identical     : True
   Differing rows           : 0  (0.00%)
   Col1 mean                : 91.8615
   Col2 mean                : 91.8615
   Col1 min / max           : 0 / 79632
   Col2 min / max           : 0 / 79632

 Friday-WorkingHours-Morning.pcap_ISCX.csv
   Are values identical     : True
   Differing rows           : 0  (0.00%)
   Col1 mean                : 324.4953
   Col2 mean                : 324.4953
   Col1 min / max           : 0 / 4369484
   Col2 min / max           : 0 / 4369484

 Monday-WorkingHours.pcap_ISCX.csv
   Are values identical     : True
   Differing rows           : 0  (0.00%)
   Col1 mean                : -1320

- The output shows that both columns have the same values so we will delete the one named "Fwd Header Length.1"

In [ ]:
for i, temp in enumerate(df_list):
    if 'Fwd Header Length.1' in temp.columns:
        df_list[i] = temp.drop(columns=['Fwd Header Length.1'])
        print(f" Dropped from df_list[{i}]")
    else:
        print(f" Not found in df_list[{i}]")

 Dropped from df_list[0]
 Dropped from df_list[1]
 Dropped from df_list[2]
 Dropped from df_list[3]
 Dropped from df_list[4]
 Dropped from df_list[5]
 Dropped from df_list[6]
 Dropped from df_list[7]


In [ ]:
for i in range(len(df_list)):
    print( os.path.basename(csv_files[i]))
    print(df_list[i].shape)
    print("=" * 65)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
(225745, 78)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
(286467, 78)
Friday-WorkingHours-Morning.pcap_ISCX.csv
(191033, 78)
Monday-WorkingHours.pcap_ISCX.csv
(529918, 78)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
(288602, 78)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
(170366, 78)
Tuesday-WorkingHours.pcap_ISCX.csv
(445909, 78)
Wednesday-workingHours.pcap_ISCX.csv
(692703, 78)


In [ ]:
df_1.dtypes

,0
Destination Port,int64
Flow Duration,int64
Total Fwd Packets,int64
Total Backward Packets,int64
Total Length of Fwd Packets,int64
...,...
Idle Mean,float64
Idle Std,float64
Idle Max,int64
Idle Min,int64


In [ ]:
df_1.head().transpose()

,0,1,2,3,4
Destination Port,54865,55054,55055,46236,54863
Flow Duration,3,109,52,34,3
Total Fwd Packets,2,1,1,1,2
Total Backward Packets,0,1,1,1,0
Total Length of Fwd Packets,12,6,6,6,12
...,...,...,...,...,...
Idle Mean,0.0,0.0,0.0,0.0,0.0
Idle Std,0.0,0.0,0.0,0.0,0.0
Idle Max,0,0,0,0,0
Idle Min,0,0,0,0,0


In [ ]:
for i in range(len(df_list)):
    print( os.path.basename(csv_files[i]))
    print(df_list[i].describe())
    print("=" * 65)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
       Destination Port  Flow Duration  Total Fwd Packets  \
count      225745.00000   2.257450e+05      225745.000000   
mean         8879.61946   1.624165e+07           4.874916   
std         19754.64740   3.152437e+07          15.422874   
min             0.00000  -1.000000e+00           1.000000   
25%            80.00000   7.118000e+04           2.000000   
50%            80.00000   1.452333e+06           3.000000   
75%            80.00000   8.805237e+06           5.000000   
max         65532.00000   1.199999e+08        1932.000000   

       Total Backward Packets  Total Length of Fwd Packets  \
count           225745.000000                225745.000000   
mean                 4.572775                   939.463346   
std                 21.755356                  3249.403484   
min                  0.000000                     0.000000   
25%                  1.000000                    26.000000   
50%                  4.000000

In [ ]:
file_summaries = []

for file in csv_files:
    print("=" * 65)
    name = os.path.basename(file)
    print(f" {name}")

    temp = pd.read_csv(file, low_memory=False)
    temp.columns = temp.columns.str.strip()

    # Basic stats
    n_rows, n_cols = temp.shape
    missing = temp.isnull().sum().sum()
    numeric = temp.select_dtypes(include=[np.number])
    inf_count = np.isinf(numeric).sum().sum()
    duplicates = temp.duplicated().sum()

    print(f"   Rows         : {n_rows:,}")
    print(f"   Columns      : {n_cols}")
    print(f"   Missing vals : {missing:,}")
    print(f"   Inf values   : {inf_count:,}")
    print(f"   Duplicates   : {duplicates:,}")
    print(f"   Labels found :")

    label_counts = temp['Label'].value_counts()
    for label, count in label_counts.items():
        pct = count / n_rows * 100
        print(f"      → {label:<40} {count:>8,}  ({pct:.2f}%)")

    file_summaries.append({
        'file': name,
        'rows': n_rows,
        'missing': missing,
        'inf_values': inf_count,
        'duplicates': duplicates,
        'n_labels': len(label_counts)
    })

print("\n" + "=" * 65)
print("SUMMARY TABLE")
print("=" * 65)
summary_df = pd.DataFrame(file_summaries)
print(summary_df.to_string(index=False))

 Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   Rows         : 225,745
   Columns      : 79
   Missing vals : 4
   Inf values   : 64
   Duplicates   : 2,633
   Labels found :
      → DDoS                                      128,027  (56.71%)
      → BENIGN                                     97,718  (43.29%)
 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   Rows         : 286,467
   Columns      : 79
   Missing vals : 15
   Inf values   : 727
   Duplicates   : 72,353
   Labels found :
      → PortScan                                  158,930  (55.48%)
      → BENIGN                                    127,537  (44.52%)
 Friday-WorkingHours-Morning.pcap_ISCX.csv
   Rows         : 191,033
   Columns      : 79
   Missing vals : 28
   Inf values   : 216
   Duplicates   : 6,888
   Labels found :
      → BENIGN                                    189,067  (98.97%)
      → Bot                                         1,966  (1.03%)
 Monday-WorkingHours.pcap_ISCX.csv
   Rows         

- we notice that accross all the files there are duplicates and since they represent a small proportion of the data we can delete them after merging all the dataframes
- We can also notice that there are some label's encoding problem like "Web Attack � Sql Injection"
- We can see that there are infinite and missing values and we need to handle them(delete? impute?)
- The class imbalance is the most critical problem that we need to deal with appropriate sampling techniques
- The following section will handle all of these

###1. merging the dataframes

In [ ]:
df = pd.concat(df_list, ignore_index=True)
print(f"Shape after merge          : {df.shape}")


Shape after merge          : (2830743, 78)


### 2. Dropping the duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Shape after drop_duplicates: {df.shape}")
print(f"Dropped                    : {before - len(df):,} rows")

Shape after drop_duplicates: (2522362, 78)
Dropped                    : 308,381 rows


###3. Fixing the label ecoding

In [ ]:
df['Label'] = (df['Label']
               .str.replace('Web Attack \ufffd Brute Force', 'Web Attack - Brute Force', regex=False)
               .str.replace('Web Attack \ufffd XSS', 'Web Attack - XSS', regex=False)
               .str.replace('Web Attack \ufffd Sql Injection', 'Web Attack - Sql Injection', regex=False)
               .str.strip())

In [ ]:
df['Label'].value_counts()

,count
Label,
BENIGN,2096484
DoS Hulk,172849
DDoS,128016
PortScan,90819
DoS GoldenEye,10286
FTP-Patator,5933
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3219


### 4. Handling infinite values



In [ ]:
# find columns that have Inf values
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_mask = np.isinf(df[numeric_cols])

inf_summary = inf_mask.sum()
inf_summary = inf_summary[inf_summary > 0].sort_values(ascending=False)

print("=== Columns Containing Inf Values ===\n")
for col, count in inf_summary.items():
    pct = count / len(df) * 100
    print(f"  {col:<35} {count:>8,}  ({pct:.4f}%)")

print(f"\nTotal inf values : {inf_mask.sum().sum():,}")
print(f"Affected columns : {len(inf_summary)}")

=== Columns Containing Inf Values ===

  Flow Packets/s                         1,564  (0.0620%)
  Flow Bytes/s                           1,211  (0.0480%)

Total inf values : 2,775
Affected columns : 2


- This is probably due to a division by zero since both columns are related to time (Flow Duration)
- In the following cell we'll check that

In [ ]:
for col in inf_summary.index:
    inf_rows = df[np.isinf(df[col])]
    print(f"--- {col} ---")
    print(f"  Count          : {len(inf_rows):,}")
    print(f"  Label distribution in inf rows:")
    for label, count in inf_rows['Label'].value_counts().items():
        print(f"    → {label:<40} {count:>6,}")
    print(f"  Flow Duration stats for these rows:")
    if 'Flow Duration' in df.columns:
        print(f"    min : {inf_rows['Flow Duration'].min()}")
        print(f"    max : {inf_rows['Flow Duration'].max()}")
        print(f"    mean: {inf_rows['Flow Duration'].mean():.4f}")
    print()

--- Flow Packets/s ---
  Count          : 1,564
  Label distribution in inf rows:
    → BENIGN                                    1,427
    → PortScan                                    125
    → Bot                                           5
    → DoS Hulk                                      3
    → DDoS                                          2
    → FTP-Patator                                   2
  Flow Duration stats for these rows:
    min : 0
    max : 0
    mean: 0.0000

--- Flow Bytes/s ---
  Count          : 1,211
  Label distribution in inf rows:
    → BENIGN                                    1,077
    → PortScan                                    125
    → Bot                                           5
    → DDoS                                          2
    → FTP-Patator                                   2
  Flow Duration stats for these rows:
    min : 0
    max : 0
    mean: 0.0000



- The results show that the inf values are directly caused by the Flow Duration that is 0 and probably are incomplete flow records and the best solution is to drop them since they don't concern only one class and they represent 2,775 rows out of ~2.5M  →  0.11% of dataset

- Deleting the rows

In [ ]:
# Drop rows where Flow Duration = 0 (root cause of all Inf values)
before = len(df)
df = df[df['Flow Duration'] != 0]
after = len(df)

print(f"Rows before : {before:,}")
print(f"Rows after  : {after:,}")
print(f"Dropped     : {before - after:,} rows  ({(before-after)/before*100:.4f}%)")

# Verify no Inf values remain
numeric_cols = df.select_dtypes(include=[np.number]).columns
remaining_inf = np.isinf(df[numeric_cols]).sum().sum()
print(f"\n Remaining Inf values : {remaining_inf}")

Rows before : 2,522,362
Rows after  : 2,520,798
Dropped     : 1,564 rows  (0.0620%)

 Remaining Inf values : 0


### 5. Handling missing values

In [ ]:
# Which columns have missing values?
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

print("=== Columns Containing Missing Values ===\n")
for col, count in missing_summary.items():
    pct = count / len(df) * 100
    print(f"  {col:<35} {count:>8,}  ({pct:.4f}%)")

print(f"\nTotal missing values : {df.isnull().sum().sum():,}")
print(f"Affected columns     : {len(missing_summary)}")

=== Columns Containing Missing Values ===


Total missing values : 0
Affected columns     : 0


- The missing values are gone after we deleted the duplicates and the infinite values

### 6. Final analysis before splitting

In [ ]:
print("=== Final Dataset Status ===\n")
print(f"Shape          : {df.shape}")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Inf values     : {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")
print(f"Duplicates     : {df.duplicated().sum()}")

print("\n=== Label Distribution ===\n")
label_counts = df['Label'].value_counts()
total = len(df)
for label, count in label_counts.items():
    pct = count / total * 100
    print(f"  {label:<45} {count:>8,}  ({pct:5.4f}%) ")

print(f"\nTotal samples : {total:,}")
print(f"Total classes : {label_counts.nunique()}")

=== Final Dataset Status ===

Shape          : (2520798, 78)
Missing values : 0
Inf values     : 0
Duplicates     : 0

=== Label Distribution ===

  BENIGN                                        2,095,057  (83.1109%) 
  DoS Hulk                                       172,846  (6.8568%) 
  DDoS                                           128,014  (5.0783%) 
  PortScan                                        90,694  (3.5978%) 
  DoS GoldenEye                                   10,286  (0.4080%) 
  FTP-Patator                                      5,931  (0.2353%) 
  DoS slowloris                                    5,385  (0.2136%) 
  DoS Slowhttptest                                 5,228  (0.2074%) 
  SSH-Patator                                      3,219  (0.1277%) 
  Bot                                              1,948  (0.0773%) 
  Web Attack - Brute Force                         1,470  (0.0583%) 
  Web Attack - XSS                                   652  (0.0259%) 
  Infiltration         

In [ ]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Destination Port,2520798.0,8.690590e+03,1.901280e+04,0.0,53.0,80.0,443.0,65535.0
Flow Duration,2520798.0,1.659161e+07,3.523276e+07,-13.0,208.0,50622.0,5333340.5,119999998.0
Total Fwd Packets,2520798.0,1.028174e+01,7.944201e+02,1.0,2.0,2.0,6.0,219759.0
Total Backward Packets,2520798.0,1.157280e+01,1.056922e+03,0.0,1.0,2.0,5.0,291922.0
Total Length of Fwd Packets,2520798.0,6.119477e+02,1.058827e+04,0.0,12.0,66.0,332.0,12900000.0
...,...,...,...,...,...,...,...,...
Active Min,2520798.0,6.546359e+04,6.111585e+05,0.0,0.0,0.0,0.0,110000000.0
Idle Mean,2520798.0,9.337367e+06,2.484818e+07,0.0,0.0,0.0,0.0,120000000.0
Idle Std,2520798.0,5.657941e+05,4.874169e+06,0.0,0.0,0.0,0.0,76900000.0
Idle Max,2520798.0,9.763770e+06,2.561746e+07,0.0,0.0,0.0,0.0,120000000.0


# 3. Splitting the Data

In [ ]:
!pip install  scikit-learn
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Label'])
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # ensures all classes are proportionally represented
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

print("\n=== Class distribution in train set ===\n")
for label, count in y_train.value_counts().items():
    print(f"  {label:<45} {count:>8,}")

print("\n=== Class distribution in test set ===\n")
for label, count in y_test.value_counts().items():
    print(f"  {label:<45} {count:>8,}")

X_train : (2016638, 77)
X_test  : (504160, 77)

=== Class distribution in train set ===

  BENIGN                                        1,676,045
  DoS Hulk                                       138,277
  DDoS                                           102,411
  PortScan                                        72,555
  DoS GoldenEye                                    8,229
  FTP-Patator                                      4,745
  DoS slowloris                                    4,308
  DoS Slowhttptest                                 4,182
  SSH-Patator                                      2,575
  Bot                                              1,558
  Web Attack - Brute Force                         1,176
  Web Attack - XSS                                   522
  Infiltration                                        29
  Web Attack - Sql Injection                          17
  Heartbleed                                           9

=== Class distribution in test set ===

  BENIGN      

### Applying BorderlineSMOTE sampling
- this will oversample the minority and undersample the majority class

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

# Step 1 — Define target sampling strategy
# Classes already above 10K are kept as is
oversample_strategy = {
    'DoS GoldenEye'              : 10000,
    'FTP-Patator'                : 10000,
    'DoS slowloris'              : 10000,
    'DoS Slowhttptest'           : 10000,
    'SSH-Patator'                : 10000,
    'Bot'                        : 10000,
    'Web Attack - Brute Force'   : 10000,
    'Web Attack - XSS'           : 10000,
    'Infiltration'               : 10000,
    'Web Attack - Sql Injection' : 10000,
    'Heartbleed'                 : 10000,
}

undersample_strategy = {
    'BENIGN' : 200000
}

# Step 2 — Apply Borderline-SMOTE first
print("Applying Borderline-SMOTE...")
bsmote = BorderlineSMOTE(
    sampling_strategy=oversample_strategy,
    k_neighbors=5,
    m_neighbors=10,
    kind='borderline-1',  # only DANGER zone samples used for generation
    random_state=42,
)
X_res, y_res = bsmote.fit_resample(X_train, y_train)
print(f"Shape after Borderline-SMOTE : {X_res.shape}")

# Step 3 — Undersample BENIGN
print("Applying Random Undersampling on BENIGN...")
rus = RandomUnderSampler(
    sampling_strategy=undersample_strategy,
    random_state=42
)
X_res, y_res = rus.fit_resample(X_res, y_res)
print(f"Shape after Undersampling    : {X_res.shape}")

# Step 4 — Verify final distribution
print("\n=== Final Train Distribution After Resampling ===\n")
from collections import Counter
counts = Counter(y_res)
total = len(y_res)
for label, count in sorted(counts.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    print(f"  {label:<45} {count:>8,}  ({pct:5.2f}%) ")
print(f"\nTotal train samples : {total:,}")

Applying Borderline-SMOTE...


note that more visualizations would be added later in the EDA section
and the last cell is not tested yet(need to free some RAM before trying to run it)